In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [8]:
df = pd.read_csv('/content/qoute_dataset.csv')

In [9]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [10]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [11]:
df.shape

(3038, 2)

In [12]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [ ]:
quotes = df['quote'].str.lower()

In [ ]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [13]:
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [15]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [16]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8213


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [17]:
sequences = tokenizer.texts_to_sequences(quotes)

In [18]:
for i in range(3):
    print(quotes[i])

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
“It is our choices, Harry, that show what we truly are, far more than our abilities.”
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”


In [19]:
for i in range(3):
    print(sequences[i])

[732, 61, 30, 19, 17, 901, 10, 7, 5, 1188, 8, 71, 290, 10, 149, 12, 835, 103, 776, 71, 290, 136]
[977, 7, 71, 902, 388, 9, 431, 21, 19, 479, 14, 299, 51, 55, 71, 2545, 136]
[1384, 14, 53, 203, 700, 3, 80, 15, 37, 36, 7, 30, 300, 92, 7, 5, 1073, 1, 104, 7, 30, 300, 126, 7, 5, 1073, 136]


In [20]:
X = []
y = []

for seq in sequences:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [21]:
len(X)

86549

In [22]:
len(y)

86549

In [23]:
max_len = max(len(x) for x in X)
print(max_len)

748


In [24]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [25]:
y = np.array(y)

In [26]:
X_padded.shape

(86549, 748)

In [27]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [28]:
y_one_hot.shape

(86549, 10000)

In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [30]:
embedding_dim = 50
rnn_units = 128

In [31]:
rnn_model = Sequential()
rnn_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len))
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [32]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [33]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len))
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [35]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [36]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
epochs = 10
batch_size = 128

In [38]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 48s 70ms/step - accuracy: 0.0444 - loss: 6.6680 - val_accuracy: 0.0608 - val_loss: 6.5181
Epoch 2/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.0731 - loss: 6.1641 - val_accuracy: 0.0821 - val_loss: 6.3656
Epoch 3/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.0964 - loss: 5.8269 - val_accuracy: 0.0968 - val_loss: 6.2566
Epoch 4/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.1124 - loss: 5.5587 - val_accuracy: 0.1002 - val_loss: 6.2494
Epoch 5/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.1248 - loss: 5.3256 - val_accuracy: 0.1051 - val_loss: 6.2768
Epoch 6/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 41s 64ms/step - accuracy: 0.1357 - loss: 5.1142 - val_accuracy: 0.1101 - val_loss: 6.3040
Epoch 7/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 39s 63ms/step - accuracy: 0.1461 - loss: 4.9148 - val_accuracy: 0.1092 - val_loss: 6.3595
Epoch 8/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 39s 63ms/step - accuracy: 0.1584 - loss: 4.7267 - 

In [39]:
epochs = 100
batch_size = 128
history_lstm = lstm_model.fit(
X_padded,
y_one_hot,
epochs=epochs,
batch_size=batch_size,
validation_split=0.1
)

Epoch 1/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 41s 57ms/step - accuracy: 0.0392 - loss: 6.7085 - val_accuracy: 0.0452 - val_loss: 6.6502
Epoch 2/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.0597 - loss: 6.2641 - val_accuracy: 0.0702 - val_loss: 6.5012
Epoch 3/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.0827 - loss: 6.0042 - val_accuracy: 0.0892 - val_loss: 6.4014
Epoch 4/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 34s 56ms/step - accuracy: 0.0987 - loss: 5.8011 - val_accuracy: 0.0967 - val_loss: 6.3636
Epoch 5/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.1092 - loss: 5.6290 - val_accuracy: 0.1003 - val_loss: 6.3534
Epoch 6/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.1190 - loss: 5.4600 - val_accuracy: 0.1028 - val_loss: 6.3558
Epoch 7/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.1281 - loss: 5.3011 - val_accuracy: 0.1062 - val_loss: 6.3750
Epoch 8/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 41s 54ms/step - accuracy: 0.1364 - loss: 5

In [40]:
lstm_model.save('lstm_model.h5')

In [41]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word [index] = word

In [42]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [47]:
def predictor(model, tokenizer, text, max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')
  pred = model.predict(seq, verbose=0)
  pred_index = np.argmax(pred)

  return index_to_word[pred_index]


In [57]:
seed_text = "why are you"
next_word = predictor(lstm_model, tokenizer, seed_text, max_len)
print(next_word)

worrying


In [60]:
def genrate_text(model, tokenizer, seed_text, max_len, n_words):
  for _ in range(n_words):
    next_word = predictor(model, tokenizer, seed_text, max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [67]:
seed  = "the meaning of life"
generated_text = genrate_text(lstm_model, tokenizer, seed, max_len, 10)
print(generated_text)

the meaning of life is not to love them when the heart will be


In [70]:
import pickle
with open ('tokenizer.pkl', 'wb') as f:
  pickle.dump(tokenizer, f)

In [71]:
with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)